# Qwen3-ASR 1.7B: does float16 cost accuracy? (development set only)

Team Tokens · COE025 Project 3 · Finnish ASR

The Qwen3-ASR baseline was run in **float16** because the Colab T4 has no native bfloat16.
This notebook checks whether that precision choice lowers accuracy, by running the same checkpoint
in **float16 and float32** on **400 development clips** (200 `dev_cv` + 200 `dev_parl`, evenly spaced
through each split). **No test clip is used**, following the rule that settings are chosen on the
development split and the test split is used once.

**Decision rule (fixed before running):** if float32 has a combined dev WER at least **1.0 point lower**
than float16, the three test sets are re-run in float32 and that run replaces the float16 result.
Otherwise the float16 test result stays and this check is reported as evidence that precision did
not matter.

Everything else is identical to `scripts/eval_qwen3_asr.ipynb`: checkpoint `Qwen/Qwen3-ASR-1.7B`,
forced `language="Finnish"`, greedy decoding, `max_new_tokens=256`, shared `normalize()` and WER/CER.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all (≈ 20–30 min).
At the end `qwen_precision_check.zip` downloads.

In [ ]:
!pip -q install -U qwen-asr datasets jiwer soundfile

In [ ]:
# Get the repo (shared evaluation code) and work inside it
import os, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"   # plain HTTP download; the Xet downloader can hang on Colab
if not os.path.isdir("/content/team-tokens-finnish-asr"):
    !git clone -q https://github.com/zehraoz36/team-tokens-finnish-asr /content/team-tokens-finnish-asr
else:
    !git -C /content/team-tokens-finnish-asr pull -q
%cd /content/team-tokens-finnish-asr
sys.path.insert(0, "scripts")
import asr_eval as E
print(E.hardware())
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU: Colab gave a CPU runtime (GPU quota used up?). "
                     "Runtime -> Change runtime type -> T4 GPU, or try later / another account.")

In [ ]:
import gc, time, importlib.metadata as md
import numpy as np
from datasets import load_dataset, Audio
from qwen_asr import Qwen3ASRModel

MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
REVISION = E.model_revision(MODEL_ID)
print("qwen-asr", md.version("qwen-asr"), "| transformers", md.version("transformers"), "| revision", REVISION)

# 200 evenly spaced clips from each development split (deterministic, no randomness)
N_PER_SPLIT = 200
dev = {}
for split in ("dev_cv", "dev_parl"):
    ds = load_dataset(E.HF_DATASET, split=split)
    idx = np.linspace(0, len(ds) - 1, N_PER_SPLIT).round().astype(int).tolist()
    dev[split] = ds.select(idx).cast_column("audio", Audio(sampling_rate=16000))
    print(f"{split}: {len(dev[split])} of {len(ds)} clips, "
          f"{sum(dev[split]['duration']) / 60:.1f} min, {len(set(dev[split]['speaker']))} speakers")

In [ ]:
def run(dtype, batch_size):
    """Transcribe both dev subsets with one precision. Returns {split: (ids, refs, raw_hyps, seconds)}."""
    model = Qwen3ASRModel.from_pretrained(MODEL_ID, dtype=dtype, device_map="cuda:0",
                                          max_inference_batch_size=batch_size, max_new_tokens=256)
    def fn(arrays):
        return [r.text for r in model.transcribe(audio=[(a, 16000) for a in arrays], language="Finnish")]
    fn([dev["dev_cv"][0]["audio"]["array"]])          # warm-up, not timed
    out = {}
    for split, ds in dev.items():
        ids, refs, raw, sec, _ = E.run_split(fn, ds, batch_size, desc=f"{dtype} {split}")
        out[split] = (ids, refs, raw, sec)
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

results = {}
results["float16"] = run(torch.float16, 16)
results["float32"] = run(torch.float32, 8)      # float32 needs about twice the memory

In [ ]:
import pandas as pd
rows, preds = [], []
for prec, out in results.items():
    all_refs, all_hyps = [], []
    for split, (ids, refs, raw, sec) in out.items():
        hyps = [E.normalize(h) for h in raw]
        all_refs += refs; all_hyps += hyps
        rows.append(dict(precision=prec, subset=split, clips=len(ids), seconds=round(sec, 1), **E.score(refs, hyps)))
        preds += [dict(precision=prec, subset=split, id=i, reference=r, hypothesis=h, raw_hypothesis=x)
                  for i, r, h, x in zip(ids, refs, hyps, raw)]
    rows.append(dict(precision=prec, subset="combined", clips=len(all_refs),
                     seconds=round(sum(v[3] for v in out.values()), 1), **E.score(all_refs, all_hyps)))
table = pd.DataFrame(rows)
display(table[["precision", "subset", "clips", "wer", "cer", "substitutions", "deletions", "insertions", "seconds"]])

wer16 = table.query("precision=='float16' and subset=='combined'").wer.item()
wer32 = table.query("precision=='float32' and subset=='combined'").wer.item()
same = sum(a["hypothesis"] == b["hypothesis"] for a, b in
           zip([p for p in preds if p["precision"] == "float16"], [p for p in preds if p["precision"] == "float32"]))
print(f"Combined dev WER: float16 {wer16:.2f} | float32 {wer32:.2f} | difference {wer16 - wer32:+.2f}")
print(f"Identical normalized outputs: {same} of {len(preds) // 2} clips")
RERUN = (wer16 - wer32) >= 1.0
print("DECISION:", "re-run the test sets in float32" if RERUN else "keep the float16 test result (difference < 1.0 WER)")

## Save and download

In [ ]:
import json, zipfile
from google.colab import files
os.makedirs("results", exist_ok=True)
table.assign(model=MODEL_ID, revision=REVISION, hardware=E.hardware(),
             qwen_asr=md.version("qwen-asr"), transformers=md.version("transformers"),
             decision_rerun_float32=RERUN).to_csv("results/qwen_precision_check.csv", index=False)
pd.DataFrame(preds).to_csv("results/qwen_precision_check_predictions.csv", index=False)
with zipfile.ZipFile("/content/qwen_precision_check.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("results/qwen_precision_check.csv")
    z.write("results/qwen_precision_check_predictions.csv")
files.download("/content/qwen_precision_check.zip")